# Chapter 03: Projective Geometry and Transformations of 3D

Source orientation: printed pages 65-86; PDF pages 83-104. The source was read for terminology, section order, and mathematical coverage; the prose, code, diagrams, checks, and artifacts here are original.

This notebook studies projective 3-space as a computational language for cameras and reconstruction. The central move is to stop treating a point, a plane, a line, or a quadric as a drawing primitive and instead represent each object by the homogeneous algebra that tells us what stays invariant under a change of 3D projective frame.


## Chapter Goal

By the end of the chapter you should be able to move between the geometric object and its array representation in `P^3`, apply a 4 by 4 projective transformation correctly, and verify the invariant that tells you whether the result is still the same geometric fact. The most important invariants in this pass are point-plane incidence, the Plucker/Klein constraint for 3D lines, quadric residuals and signatures, the fixed plane at infinity for affine structure, and the absolute conic or absolute dual quadric for metric structure.


## Computational Translation Guide

- A finite 3D point is stored as `X = (x, y, z, 1)`. A direction or ideal point has last coordinate zero. Equality is always up to nonzero scale.
- A plane is a homogeneous 4-vector `pi`; a point lies on it exactly when `pi.T @ X = 0`.
- A projective transformation sends points by `X_prime = H @ X`; the matching plane update is `pi_prime = inv(H).T @ pi`.
- A line in `P^3` cannot be replaced by one cross product. We use a 4 by 4 skew Plucker matrix `L = A B.T - B A.T` and its six Plucker coordinates. Valid line coordinates lie on the Klein quadric.
- A point quadric is a symmetric matrix `Q` with `X.T @ Q @ X = 0`. A dual quadric is the corresponding equation on tangent planes.
- The plane at infinity `pi_inf` is the affine upgrade object. The absolute conic `Omega_inf` and the absolute dual quadric `Qstar_inf` are the metric upgrade objects.


## Library Routing

| Chapter concept | Representation | Library | Why this route | Artifact/check |
| --- | --- | --- | --- | --- |
| Point-plane incidence and projective transport | Interactive 3D planes, points, and transformed copies | Plotly + NumPy | The learner can rotate a genuine 3D homogeneous construction while NumPy checks `pi.T @ X` before and after `H` | `point-plane-incidence-transform.html`, incidence residuals |
| 3D lines and Plucker coordinates | Static 3D line scene plus numeric Plucker table | Matplotlib + NumPy | The visual distinguishes intersecting from skew lines, while the six-coordinate Klein equation catches invalid line data | `plucker-skew-lines-klein-constraint.png`, Klein and bilinear products |
| Quadrics, ruled generators, and twisted cubic | Interactive surface, generator lines, conic slice, and space curve | Plotly + SymPy | Plotly exposes 3D ruled structure; SymPy verifies the cubic plane-intersection scaffold exactly | `quadric-rulings-twisted-cubic.html`, residuals and symbolic degree |
| Plane at infinity, absolute conic, and absolute dual quadric | Polarity diagram and metric-invariant bar chart | Matplotlib + NumPy | The absolute conic has no real point locus, so a direction-polar diagram and `Qstar_inf` angle check are more honest than a fake real conic | `absolute-conic-polarity-and-dual-quadric.png`, angle/null-vector checks |
| Quadric classification | Rank/signature table | Pandas + NumPy | Rank and signature are algebraic labels that should be auditable as data, not embedded in prose only | `quadric-signature-classification.csv` |


## Route Through The Chapter

1. Build the point-plane dictionary for `P^3` and verify that incidence survives a 3D projective transformation.
2. Replace the tempting but wrong single-cross-product model of 3D lines with Plucker matrices and the Klein quadric constraint.
3. Read quadrics as symmetric matrices: samples must satisfy an implicit equation, and rank/signature explain projective class.
4. Add the twisted cubic as a space curve whose plane intersections are cubic rather than planar.
5. Separate projective, affine, and metric structure by tracking `pi_inf`, `Omega_inf`, and `Qstar_inf`.
6. Finish with a metric-upgrade lab: compare naive transformed normals against the invariant absolute-dual-quadric angle formula.


In [ ]:
from pathlib import Path
import sys
import math

import numpy as np
import pandas as pd
import sympy as sp

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

import plotly.graph_objects as go
from plotly.subplots import make_subplots

CURRENT = Path.cwd().resolve()
for candidate in [CURRENT, *CURRENT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the MVG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (
    assert_artifacts,
    display_artifact,
    relative_to_book,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)

TOPIC = "chapter-03"
artifact_paths: list[Path] = []
visual_artifact_paths: list[Path] = []
check_payload: dict[str, object] = {}
np.set_printoptions(precision=5, suppress=True)


In [ ]:
def hpoint(x, y, z, w=1.0):
    return np.array([x, y, z, w], dtype=float)


def dehomogeneous(X):
    X = np.asarray(X, dtype=float)
    if abs(X[-1]) < 1e-12:
        raise ValueError("point is at infinity in this affine chart")
    return X[:3] / X[-1]


def dehomogeneous_many(points):
    return np.vstack([dehomogeneous(point) for point in points])


def normalize_homogeneous(v):
    v = np.asarray(v, dtype=float)
    scale = np.linalg.norm(v)
    if scale == 0:
        raise ValueError("zero homogeneous vector")
    v = v / scale
    pivot = np.flatnonzero(np.abs(v) > 1e-12)
    if len(pivot) and v[pivot[0]] < 0:
        v = -v
    return v


def null_vector(M):
    _, _, vh = np.linalg.svd(np.asarray(M, dtype=float))
    return normalize_homogeneous(vh[-1])


def plane_from_points(A, B, C):
    return null_vector(np.vstack([A, B, C]))


def transform_plane(H, plane):
    return np.linalg.inv(H).T @ plane


def proportional_residual(a, b):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return float("inf")
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return float(min(np.linalg.norm(a - b), np.linalg.norm(a + b)))


def plane_patch(plane, center, extent=0.9, samples=12):
    normal = np.asarray(plane[:3], dtype=float)
    normal = normal / np.linalg.norm(normal)
    seed = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(seed, normal)) > 0.82:
        seed = np.array([0.0, 1.0, 0.0])
    u = np.cross(normal, seed)
    u = u / np.linalg.norm(u)
    v = np.cross(normal, u)
    grid = np.linspace(-extent, extent, samples)
    X, Y = np.meshgrid(grid, grid)
    P = center[:, None, None] + X[None, :, :] * u[:, None, None] + Y[None, :, :] * v[:, None, None]
    return P[0], P[1], P[2]


def line_plucker_matrix_from_points(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    return np.outer(A, B) - np.outer(B, A)


def plucker_coords_from_matrix(L):
    return np.array([L[0, 1], L[0, 2], L[0, 3], L[1, 2], L[3, 1], L[2, 3]], dtype=float)


def plucker_matrix_from_coords(coords):
    l12, l13, l14, l23, l42, l34 = coords
    L = np.zeros((4, 4), dtype=float)
    L[0, 1], L[1, 0] = l12, -l12
    L[0, 2], L[2, 0] = l13, -l13
    L[0, 3], L[3, 0] = l14, -l14
    L[1, 2], L[2, 1] = l23, -l23
    L[3, 1], L[1, 3] = l42, -l42
    L[2, 3], L[3, 2] = l34, -l34
    return L


def dual_plucker_matrix(L):
    l12, l13, l14, l23, l42, l34 = plucker_coords_from_matrix(L)
    return plucker_matrix_from_coords([l34, l42, l23, l14, l13, l12])


def klein_constraint(coords):
    l12, l13, l14, l23, l42, l34 = coords
    return float(l12 * l34 + l13 * l42 + l14 * l23)


def line_bilinear(coords_a, coords_b):
    a12, a13, a14, a23, a42, a34 = coords_a
    b12, b13, b14, b23, b42, b34 = coords_b
    return float(
        a12 * b34 + b12 * a34
        + a13 * b42 + b13 * a42
        + a14 * b23 + b14 * a23
    )


def line_points_for_plot(A, B, t0=-0.35, t1=1.35, samples=30):
    ts = np.linspace(t0, t1, samples)
    affine_a = dehomogeneous(A)
    affine_b = dehomogeneous(B)
    return affine_a[None, :] + ts[:, None] * (affine_b - affine_a)[None, :]


def quadric_residual(Q, points):
    values = [float(P.T @ Q @ P) for P in points]
    return max(abs(v) for v in values), values


def rank_signature(D):
    eig = np.linalg.eigvalsh(np.asarray(D, dtype=float))
    rank = int(np.sum(np.abs(eig) > 1e-10))
    positive = int(np.sum(eig > 1e-10))
    negative = int(np.sum(eig < -1e-10))
    signature = abs(positive - negative)
    return rank, signature


def angle_cosine_from_dual_quadric(pi1, pi2, Qstar):
    numerator = float(pi1.T @ Qstar @ pi2)
    denom = math.sqrt(float(pi1.T @ Qstar @ pi1) * float(pi2.T @ Qstar @ pi2))
    return numerator / denom


## 1. Points, Planes, And Projective Transport

The first artifact, `point-plane-incidence-transform.html`, is an incidence audit in 3D. The blue triangle defines a plane from three homogeneous points. A projective matrix sends the three points to a new triangle and sends the plane by the inverse-transpose rule. The inspection target is simple: the transformed points should still sit on the transformed plane, even though the affine appearance of the plane has changed.


In [ ]:
A = hpoint(0.0, 0.0, 0.0)
B = hpoint(1.15, 0.12, 0.18)
C = hpoint(0.18, 1.05, 0.42)
D = 0.25 * A + 0.35 * B + 0.40 * C
plane = plane_from_points(A, B, C)

H = np.array(
    [
        [1.15, 0.22, -0.10, 0.35],
        [0.08, 0.92, 0.26, -0.20],
        [-0.06, 0.18, 1.08, 0.28],
        [0.10, -0.06, 0.04, 1.00],
    ],
    dtype=float,
)
transformed = [H @ P for P in [A, B, C, D]]
plane_transformed = transform_plane(H, plane)

original_points = dehomogeneous_many([A, B, C, D])
transformed_points = dehomogeneous_many(transformed)
PX, PY, PZ = plane_patch(plane, original_points[:3].mean(axis=0), extent=0.82)
QX, QY, QZ = plane_patch(plane_transformed, transformed_points[:3].mean(axis=0), extent=0.82)

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("Original chart", "After X_prime = H X"),
)
fig.add_trace(go.Surface(x=PX, y=PY, z=PZ, opacity=0.35, showscale=False, colorscale=[[0, "#dbeafe"], [1, "#93c5fd"]]), row=1, col=1)
fig.add_trace(go.Scatter3d(x=original_points[:, 0], y=original_points[:, 1], z=original_points[:, 2], mode="markers+text", text=["A", "B", "C", "D"], textposition="top center", marker=dict(size=5, color="#1d4ed8")), row=1, col=1)
fig.add_trace(go.Mesh3d(x=original_points[:3, 0], y=original_points[:3, 1], z=original_points[:3, 2], i=[0], j=[1], k=[2], opacity=0.28, color="#2563eb"), row=1, col=1)
fig.add_trace(go.Surface(x=QX, y=QY, z=QZ, opacity=0.35, showscale=False, colorscale=[[0, "#fee2e2"], [1, "#fca5a5"]]), row=1, col=2)
fig.add_trace(go.Scatter3d(x=transformed_points[:, 0], y=transformed_points[:, 1], z=transformed_points[:, 2], mode="markers+text", text=["H A", "H B", "H C", "H D"], textposition="top center", marker=dict(size=5, color="#b91c1c")), row=1, col=2)
fig.add_trace(go.Mesh3d(x=transformed_points[:3, 0], y=transformed_points[:3, 1], z=transformed_points[:3, 2], i=[0], j=[1], k=[2], opacity=0.28, color="#dc2626"), row=1, col=2)
fig.update_layout(title="Point-plane incidence is a projective statement", height=620, margin=dict(l=10, r=10, t=70, b=10), showlegend=False)
for scene_id in ["scene", "scene2"]:
    fig.update_layout({scene_id: dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z")})

incidence_path = save_plotly_html(fig, TOPIC, "interactive", "point-plane-incidence-transform.html")
artifact_paths.append(incidence_path)
visual_artifact_paths.append(incidence_path)
display_artifact(incidence_path, width=920, height=640)

original_incidence = [float(plane @ P) for P in [A, B, C, D]]
transformed_incidence = [float(plane_transformed @ P) for P in transformed]
check_payload["point_plane_incidence"] = {
    "max_original_incidence_abs": max(abs(v) for v in original_incidence),
    "max_transformed_incidence_abs": max(abs(v) for v in transformed_incidence),
    "det_H": float(np.linalg.det(H)),
}
check_payload["point_plane_incidence"]


The determinant of `H` is nonzero, so the transformation is a valid collineation of projective 3-space. The check payload records that the incidence residuals stay at numerical zero. This is the prototype for the rest of the chapter: a geometric statement is represented by a scalar that should not change except for homogeneous scale.


## 2. Lines Need Plucker Coordinates

In the projective plane, two points give a line by one cross product. In `P^3`, two points still define a line, but the minimal algebra is a skew 4 by 4 Plucker matrix with six independent entries constrained by one quadratic equation. The artifact `plucker-skew-lines-klein-constraint.png` shows one line, a line that intersects it, and a line that is skew to it. The inspection target is the bilinear Plucker product: zero means the two 3D lines are coplanar and therefore intersect; nonzero means skew.


In [ ]:
L1_A = hpoint(0.0, 0.0, 0.0)
L1_B = hpoint(1.0, 0.22, 0.12)
L2_A = hpoint(0.10, 0.90, -0.15)
L2_B = hpoint(1.10, 1.12, 0.82)
meet = 0.55 * L1_A + 0.45 * L1_B
L3_B = hpoint(0.30, 0.95, 0.58)

L1 = line_plucker_matrix_from_points(L1_A, L1_B)
L2 = line_plucker_matrix_from_points(L2_A, L2_B)
L3 = line_plucker_matrix_from_points(meet, L3_B)
coords1 = plucker_coords_from_matrix(L1)
coords2 = plucker_coords_from_matrix(L2)
coords3 = plucker_coords_from_matrix(L3)

parallel_plane_1 = np.array([0.0, 0.0, 1.0, -0.40])
parallel_plane_2 = np.array([0.0, 0.0, 1.0, 0.75])
Lstar_parallel = np.outer(parallel_plane_1, parallel_plane_2) - np.outer(parallel_plane_2, parallel_plane_1)
L_parallel = dual_plucker_matrix(Lstar_parallel)
pi_infinity = np.array([0.0, 0.0, 0.0, 1.0])
parallel_intersection_at_infinity = float(np.linalg.norm(L_parallel @ pi_infinity))

fig = plt.figure(figsize=(10.5, 5.5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
for label, start, end, color in [
    ("L1", L1_A, L1_B, "#1d4ed8"),
    ("L2 skew to L1", L2_A, L2_B, "#b91c1c"),
    ("L3 meets L1", meet, L3_B, "#047857"),
]:
    pts = line_points_for_plot(start, end)
    ax.plot(pts[:, 0], pts[:, 1], pts[:, 2], color=color, linewidth=2.5, label=label)
    ax.scatter([dehomogeneous(start)[0], dehomogeneous(end)[0]], [dehomogeneous(start)[1], dehomogeneous(end)[1]], [dehomogeneous(start)[2], dehomogeneous(end)[2]], color=color, s=28)
ax.scatter(*dehomogeneous(meet), color="#111827", s=45, label="shared point")
ax.set_title("Intersecting versus skew lines in P3")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.legend(loc="upper left", fontsize=8)
ax.view_init(elev=22, azim=-55)

ax2 = fig.add_subplot(1, 2, 2)
rows = [
    ("Klein(L1)", klein_constraint(coords1)),
    ("Klein(L2)", klein_constraint(coords2)),
    ("Klein(L3)", klein_constraint(coords3)),
    ("(L1 | L3)", line_bilinear(coords1, coords3)),
    ("(L1 | L2)", line_bilinear(coords1, coords2)),
]
colors = ["#6b7280", "#6b7280", "#6b7280", "#047857", "#b91c1c"]
y = np.arange(len(rows))
ax2.axvline(0.0, color="#111827", linewidth=1)
ax2.barh(y, [value for _, value in rows], color=colors, alpha=0.82)
ax2.set_yticks(y, [name for name, _ in rows])
ax2.invert_yaxis()
ax2.set_title("Line validity and intersection tests")
ax2.set_xlabel("Plucker scalar")
for idx, (_, value) in enumerate(rows):
    ax2.text(value + (0.015 if value >= 0 else -0.015), idx, f"{value:.3g}", va="center", ha="left" if value >= 0 else "right")
ax2.text(0.02, 0.08, "Valid lines lie on the Klein quadric./nA zero bilinear product detects coplanar lines.", transform=ax2.transAxes, fontsize=9, bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#d1d5db"))
fig.tight_layout()

plucker_path = save_matplotlib(fig, TOPIC, "figures", "plucker-skew-lines-klein-constraint.png", dpi=180)
plt.close(fig)
artifact_paths.append(plucker_path)
visual_artifact_paths.append(plucker_path)
display_artifact(plucker_path, width=880)

check_payload["plucker_lines"] = {
    "klein_residuals": [klein_constraint(c) for c in [coords1, coords2, coords3]],
    "intersecting_line_bilinear_L1_L3": line_bilinear(coords1, coords3),
    "skew_line_bilinear_L1_L2": line_bilinear(coords1, coords2),
    "parallel_plane_intersection_on_pi_infinity_residual": parallel_intersection_at_infinity,
}
check_payload["plucker_lines"]


The same Plucker matrix handles joins and incidences. If `L` is the point Plucker matrix, then `L @ pi` gives the point where the line meets a plane; if that vector vanishes, the line lies in the plane. Its dual matrix gives the plane through a point and the line. The final scalar in the check payload tests a source-span claim about affine structure: two parallel planes meet in a line contained in `pi_inf`.


## 3. Quadrics, Rulings, And The Twisted Cubic

A quadric is an implicit homogeneous equation `X.T @ Q @ X = 0`. The interactive artifact `quadric-rulings-twisted-cubic.html` shows a ruled hyperboloid, two families of generator lines, a plane slice that appears as a conic, and a twisted cubic in the same affine chart. Rotate the scene and inspect that the generator curves are not decorative: every sampled point on every displayed generator satisfies the same quadric equation.


In [ ]:
Q_hyperboloid = np.diag([1.0, 1.0, -1.0, -1.0])

u = np.linspace(0, 2 * np.pi, 80)
z = np.linspace(-1.45, 1.45, 42)
U, Z = np.meshgrid(u, z)
R = np.sqrt(1 + Z**2)
X = R * np.cos(U)
Y = R * np.sin(U)

fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=Z, opacity=0.45, colorscale="Viridis", showscale=False, name="x^2+y^2-z^2=1"))

s_values = np.linspace(-1.55, 1.55, 70)
line_thetas = np.linspace(0, 2 * np.pi, 7, endpoint=False)
generator_points = []
for theta in line_thetas:
    x1 = np.cos(theta) - s_values * np.sin(theta)
    y1 = np.sin(theta) + s_values * np.cos(theta)
    z1 = s_values
    x2 = np.cos(theta) + s_values * np.sin(theta)
    y2 = np.sin(theta) - s_values * np.cos(theta)
    z2 = s_values
    generator_points.extend([hpoint(x, y, z) for x, y, z in zip(x1, y1, z1)])
    generator_points.extend([hpoint(x, y, z) for x, y, z in zip(x2, y2, z2)])
    show_ruling_legend = bool(theta == line_thetas[0])
    fig.add_trace(go.Scatter3d(x=x1, y=y1, z=z1, mode="lines", line=dict(color="#f97316", width=5), name="ruling family A", showlegend=show_ruling_legend))
    fig.add_trace(go.Scatter3d(x=x2, y=y2, z=z2, mode="lines", line=dict(color="#2563eb", width=5), name="ruling family B", showlegend=show_ruling_legend))

slice_z = 0.55
slice_r = math.sqrt(1 + slice_z**2)
fig.add_trace(go.Scatter3d(x=slice_r * np.cos(u), y=slice_r * np.sin(u), z=np.full_like(u, slice_z), mode="lines", line=dict(color="#111827", width=6), name="plane slice conic"))

t = np.linspace(-1.25, 1.25, 160)
twisted = np.vstack([t**3, t**2, t]).T
fig.add_trace(go.Scatter3d(x=twisted[:, 0], y=twisted[:, 1], z=twisted[:, 2], mode="lines", line=dict(color="#be185d", width=7), name="twisted cubic (t^3,t^2,t,1)"))
fig.update_layout(title="Ruled quadric, conic slice, and twisted cubic", height=720, scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"), margin=dict(l=0, r=0, t=55, b=0))

quadric_path = save_plotly_html(fig, TOPIC, "interactive", "quadric-rulings-twisted-cubic.html")
artifact_paths.append(quadric_path)
visual_artifact_paths.append(quadric_path)
display_artifact(quadric_path, width=920, height=720)

surface_samples = [hpoint(X[i, j], Y[i, j], Z[i, j]) for i in range(0, X.shape[0], 8) for j in range(0, X.shape[1], 13)]
max_surface_residual, _ = quadric_residual(Q_hyperboloid, surface_samples)
max_generator_residual, _ = quadric_residual(Q_hyperboloid, generator_points[::19])

sym_t = sp.symbols("t")
twisted_point = sp.Matrix([sym_t**3, sym_t**2, sym_t, 1])
probe_plane = sp.Matrix([1, sp.Rational(-1, 2), sp.Rational(1, 4), sp.Rational(-1, 10)])
intersection_polynomial = sp.expand((probe_plane.T * twisted_point)[0])

sample_t = np.array([-1.0, -0.35, 0.2, 0.9])
twisted_homogeneous_samples = np.vstack([sample_t**3, sample_t**2, sample_t, np.ones_like(sample_t)]).T
check_payload["quadrics_and_twisted_cubic"] = {
    "max_hyperboloid_surface_residual": max_surface_residual,
    "max_generator_line_residual": max_generator_residual,
    "twisted_cubic_sample_rank": int(np.linalg.matrix_rank(twisted_homogeneous_samples)),
    "twisted_cubic_general_plane_polynomial": str(intersection_polynomial),
    "twisted_cubic_general_plane_degree": int(sp.degree(intersection_polynomial, gen=sym_t)),
}
check_payload["quadrics_and_twisted_cubic"]


### Quadric Signature Table

The source span classifies point quadrics by rank and signature. The notebook records the same idea as a small computable table, saved as `quadric-signature-classification.csv`. The useful habit is to compute the label from eigenvalues and then read the geometry: nondegenerate rank-four representatives split into the sphere-type and one-sheet-hyperboloid-type cases, while lower rank cases become degenerate objects such as cones, plane pairs, lines, or a double plane.


In [ ]:
quadric_representatives = [
    {"representative": "empty_real_rank4", "diagonal": [1, 1, 1, 1], "interpretation": "no real affine points"},
    {"representative": "sphere_type", "diagonal": [1, 1, 1, -1], "interpretation": "sphere or ellipsoid class"},
    {"representative": "one_sheet_hyperboloid_type", "diagonal": [1, 1, -1, -1], "interpretation": "ruled nondegenerate class"},
    {"representative": "cone_type", "diagonal": [1, 1, -1, 0], "interpretation": "rank-three cone"},
    {"representative": "two_plane_type", "diagonal": [1, -1, 0, 0], "interpretation": "pair of planes"},
    {"representative": "double_plane_type", "diagonal": [1, 0, 0, 0], "interpretation": "double plane"},
]
for row in quadric_representatives:
    rank, signature = rank_signature(np.diag(row["diagonal"]))
    row["rank"] = rank
    row["signature"] = signature

quadric_table = pd.DataFrame(quadric_representatives)[["representative", "diagonal", "rank", "signature", "interpretation"]]
table_path = save_csv(quadric_table.to_dict(orient="records"), TOPIC, "tables", "quadric-signature-classification.csv")
artifact_paths.append(table_path)
display_artifact(table_path)
check_payload["quadric_signature_table"] = {
    "row_count": int(len(quadric_table)),
    "rank_signature_pairs": [[int(row.rank), int(row.signature)] for row in quadric_table.itertuples()],
}
quadric_table


The twisted cubic check is deliberately algebraic. Four samples of `(t^3,t^2,t,1)` have rank four, so the curve is not trapped in a plane. Intersecting it with a general plane produces a cubic polynomial in the parameter, which is the computational version of the chapter's claim that a general plane cuts the curve in three points over the algebraic closure.


## 4. The Plane At Infinity And The Absolute Conic

The affine upgrade object is `pi_inf`. An affine transformation fixes it as a set; a general projective transformation moves it. The metric upgrade object can be represented either by the absolute conic on `pi_inf` or, more conveniently for 3D plane angles, by the absolute dual quadric. The artifact `absolute-conic-polarity-and-dual-quadric.png` uses two views: a polarity diagram in a real chart of directions, and a bar chart comparing an honest `Qstar_inf` angle computation with the misleading angle obtained from transformed coordinate normals.


In [ ]:
Omega_inf = np.eye(3)
Qstar_inf = np.diag([1.0, 1.0, 1.0, 0.0])
pi_inf = np.array([0.0, 0.0, 0.0, 1.0])

d1 = np.array([1.0, 0.0, 0.0])
d2 = np.array([0.0, 1.0, 0.0])
d_bad = np.array([1.0, 0.8, 0.3])
orthogonality_residual = float(d1.T @ Omega_inf @ d2)

plane_1 = np.array([1.0, 0.45, 0.20, -0.30])
plane_2 = np.array([-0.15, 0.90, 0.55, 0.20])
cos_original = angle_cosine_from_dual_quadric(plane_1, plane_2, Qstar_inf)

H_metric = np.array(
    [
        [1.20, 0.10, -0.18, 0.35],
        [0.18, 0.95, 0.24, -0.15],
        [-0.08, 0.20, 1.10, 0.25],
        [0.11, -0.07, 0.06, 1.00],
    ],
    dtype=float,
)
plane_1_prime = transform_plane(H_metric, plane_1)
plane_2_prime = transform_plane(H_metric, plane_2)
Qstar_prime = H_metric @ Qstar_inf @ H_metric.T
pi_inf_prime = transform_plane(H_metric, pi_inf)
cos_projective = angle_cosine_from_dual_quadric(plane_1_prime, plane_2_prime, Qstar_prime)
cos_naive_normals = float(
    plane_1_prime[:3].T @ plane_2_prime[:3]
    / math.sqrt((plane_1_prime[:3].T @ plane_1_prime[:3]) * (plane_2_prime[:3].T @ plane_2_prime[:3]))
)

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.8))
ax = axes[0]
ax.set_title("Absolute-conic polarity in a direction chart")
ax.axhline(0, color="#d1d5db", linewidth=1)
ax.axvline(0, color="#111827", linewidth=2, label="polar line of d=(1,0,0)")
ax.scatter([0], [1], color="#047857", s=70, label="orthogonal direction")
ax.scatter([0], [-1], color="#047857", s=70)
ax.scatter([0.8 / 0.3], [1.0 / 0.3], color="#b91c1c", s=70, label="not on polar line")
ax.annotate("d2 satisfies d1.T Omega d2 = 0", xy=(0, 1), xytext=(0.25, 1.25), arrowprops=dict(arrowstyle="->", color="#047857"), fontsize=9)
ax.annotate("real chart of pi_inf; Omega has no real point locus", xy=(0.18, -1.2), fontsize=9)
ax.set_xlim(-2.4, 3.2)
ax.set_ylim(-2.2, 4.2)
ax.set_xlabel("direction chart x/z")
ax.set_ylabel("direction chart y/z")
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="upper right", fontsize=8)

ax = axes[1]
angle_values = [cos_original, cos_projective, cos_naive_normals]
labels = ["original/nQstar", "projective frame/nQstar prime", "naive transformed/nnormals"]
ax.bar(labels, angle_values, color=["#1d4ed8", "#047857", "#b91c1c"], alpha=0.84)
ax.axhline(cos_original, color="#111827", linestyle="--", linewidth=1)
ax.set_ylim(min(angle_values) - 0.25, max(angle_values) + 0.25)
ax.set_title("Plane-angle invariant uses the dual quadric")
ax.set_ylabel("cosine of angle")
for i, value in enumerate(angle_values):
    ax.text(i, value + 0.025, f"{value:.4f}", ha="center", va="bottom", fontsize=9)
fig.tight_layout()

absolute_path = save_matplotlib(fig, TOPIC, "figures", "absolute-conic-polarity-and-dual-quadric.png", dpi=180)
plt.close(fig)
artifact_paths.append(absolute_path)
visual_artifact_paths.append(absolute_path)
display_artifact(absolute_path, width=900)

s = 1.7
Rz = np.array([[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0]])
H_similarity = np.block([[s * Rz, np.array([[0.3], [-0.2], [0.5]])], [np.zeros((1, 3)), np.ones((1, 1))]])
Qstar_similarity = H_similarity @ Qstar_inf @ H_similarity.T
H_affine = np.block([[np.array([[1.1, 0.2, 0.0], [0.0, 0.9, 0.1], [0.1, 0.0, 1.2]]), np.array([[0.1], [0.2], [-0.1]])], [np.zeros((1, 3)), np.ones((1, 1))]])

check_payload["absolute_conic_and_dual_quadric"] = {
    "direction_orthogonality_residual": orthogonality_residual,
    "bad_direction_conjugacy_value": float(d1.T @ Omega_inf @ d_bad),
    "original_plane_angle_cosine": cos_original,
    "projective_frame_plane_angle_cosine": cos_projective,
    "naive_transformed_normal_cosine": cos_naive_normals,
    "dual_quadric_angle_error": abs(cos_original - cos_projective),
    "Qstar_pi_infinity_null_residual": float(np.linalg.norm(Qstar_inf @ pi_inf)),
    "transformed_Qstar_transformed_pi_infinity_null_residual": float(np.linalg.norm(Qstar_prime @ pi_inf_prime)),
    "similarity_Qstar_proportional_residual": proportional_residual(Qstar_similarity, Qstar_inf),
    "affine_pi_infinity_fixed_residual": proportional_residual(transform_plane(H_affine, pi_inf), pi_inf),
    "general_projective_pi_infinity_moved_residual": proportional_residual(transform_plane(H_metric, pi_inf), pi_inf),
}
check_payload["absolute_conic_and_dual_quadric"]


The point of this diagram is not to pretend that the absolute conic is visible as a real circle. In a metric frame it is a conic of complex directions on the plane at infinity. What is visible and computable is its polarity: a direction and its polar line encode orthogonality. For plane angles, the absolute dual quadric is the compact 4 by 4 object that carries both `pi_inf` and the metric conic into one equation.


## Applied Lab: Metric Upgrade In A Projective Frame

The lab repeats a common reconstruction situation in miniature. We start with Euclidean planes, apply a projective change of coordinates, and then try to measure an angle. If we use only the first three coefficients of the transformed planes as if they were Euclidean normals, the result is wrong. If we carry the transformed absolute dual quadric, the angle returns to the original value. The summary is saved as `metric-upgrade-lab-summary.json`.


In [ ]:
lab_summary = {
    "source_frame_cosine": cos_original,
    "projective_frame_cosine_using_Qstar_prime": cos_projective,
    "naive_projective_coordinate_normal_cosine": cos_naive_normals,
    "absolute_error_using_Qstar_prime": abs(cos_original - cos_projective),
    "absolute_error_using_naive_normals": abs(cos_original - cos_naive_normals),
    "rank_Qstar_prime": int(np.linalg.matrix_rank(Qstar_prime, tol=1e-10)),
    "pi_infinity_from_Qstar_prime_null_residual": float(np.linalg.norm(Qstar_prime @ pi_inf_prime)),
    "lesson": "Metric measurements in a projective reconstruction require the upgraded absolute dual quadric, not raw coordinate normals.",
}
lab_path = save_json(lab_summary, TOPIC, "checks", "metric-upgrade-lab-summary.json")
artifact_paths.append(lab_path)
display_artifact(lab_path)
lab_summary


## Pitfalls And Failure Modes

- A homogeneous vector is not valid because it has the right length. The invariant must also hold: incidence zero, nonzero determinant for a projective transformation, or the Klein equation for a line.
- A pair of finite endpoints over-parameterizes a line. Plucker coordinates remove the choice of endpoints but add the Klein constraint.
- A plotted quadric can look plausible while violating its implicit equation. Always check `X.T @ Q @ X` on displayed samples.
- A transformed plane's first three coefficients are not Euclidean normals unless the frame has already been upgraded to a metric frame.
- The absolute conic is a metric device on the plane at infinity. It is useful through conjugacy, polarity, and the absolute dual quadric, not because it gives a real drawable circle.


## Final Sanity Checks

The final cell checks core identities, artifact integrity, and named artifact paths. It also writes `projective-3d-invariant-checks.json`, which is a compact rubric for the chapter: each saved visual has a matching numeric or symbolic invariant.


In [ ]:
assert check_payload["point_plane_incidence"]["det_H"] != 0
assert check_payload["point_plane_incidence"]["max_original_incidence_abs"] < 1e-10
assert check_payload["point_plane_incidence"]["max_transformed_incidence_abs"] < 1e-10

assert max(abs(v) for v in check_payload["plucker_lines"]["klein_residuals"]) < 1e-10
assert abs(check_payload["plucker_lines"]["intersecting_line_bilinear_L1_L3"]) < 1e-10
assert abs(check_payload["plucker_lines"]["skew_line_bilinear_L1_L2"]) > 1e-3
assert check_payload["plucker_lines"]["parallel_plane_intersection_on_pi_infinity_residual"] < 1e-10

assert check_payload["quadrics_and_twisted_cubic"]["max_hyperboloid_surface_residual"] < 1e-10
assert check_payload["quadrics_and_twisted_cubic"]["max_generator_line_residual"] < 1e-10
assert check_payload["quadrics_and_twisted_cubic"]["twisted_cubic_sample_rank"] == 4
assert check_payload["quadrics_and_twisted_cubic"]["twisted_cubic_general_plane_degree"] == 3

assert check_payload["absolute_conic_and_dual_quadric"]["direction_orthogonality_residual"] == 0.0
assert check_payload["absolute_conic_and_dual_quadric"]["dual_quadric_angle_error"] < 1e-10
assert check_payload["absolute_conic_and_dual_quadric"]["Qstar_pi_infinity_null_residual"] < 1e-12
assert check_payload["absolute_conic_and_dual_quadric"]["transformed_Qstar_transformed_pi_infinity_null_residual"] < 1e-10
assert check_payload["absolute_conic_and_dual_quadric"]["similarity_Qstar_proportional_residual"] < 1e-12
assert check_payload["absolute_conic_and_dual_quadric"]["affine_pi_infinity_fixed_residual"] < 1e-12
assert check_payload["absolute_conic_and_dual_quadric"]["general_projective_pi_infinity_moved_residual"] > 1e-3
assert lab_summary["rank_Qstar_prime"] == 3
assert lab_summary["absolute_error_using_Qstar_prime"] < 1e-10

final_sanity = {
    "source_span": "printed pages 65-86; PDF pages 83-104",
    "visual_artifact_count": len(visual_artifact_paths),
    "artifact_count_before_sanity_file": len(artifact_paths),
    "visual_artifacts": [relative_to_book(path) for path in visual_artifact_paths],
    "support_artifacts": [relative_to_book(path) for path in artifact_paths if path not in visual_artifact_paths],
    "checks": check_payload,
}
sanity_path = save_json(final_sanity, TOPIC, "checks", "projective-3d-invariant-checks.json")
artifact_paths.append(sanity_path)
final_sanity["support_artifacts"].append(relative_to_book(sanity_path))
save_json(final_sanity, TOPIC, "checks", "projective-3d-invariant-checks.json")

assert_artifacts(visual_artifact_paths, min_bytes=1500)
assert_artifacts([path for path in artifact_paths if path not in visual_artifact_paths], min_bytes=80)

display_artifact(sanity_path)
final_sanity


## Takeaways

Projective 3D geometry is not just 2D projective geometry with one more coordinate. Points and planes remain dual, but lines become self-dual objects with Plucker constraints; quadrics become symmetric 4 by 4 forms; and metric meaning is no longer in the coordinate axes unless the plane at infinity and the absolute conic or absolute dual quadric have been identified. The reliable workflow is always the same: choose the representation, make the visual inspectable, and then check the invariant that the representation promises to preserve.
